# 01: System Overview — From Mathematical Framework to Implementation

**What this examines:** Selected mathematical symbols, concrete record fields and a synthetic data-flow trace. Field correspondence is narrower than proof that the implementation satisfies the full theory.

**Inputs:** None (educational notebook demonstrating architecture)

**Outputs:** Symbol-to-code mapping tables, architecture diagram, traced data flow examples

**Runtime:** ~3 min

**Goal:** Trace the selected concepts into their native owners, inspect instantiated records and follow actual write/prune events.

## 1. Setup

In [ ]:
# Run mode: "demo" (quick) or "paper" (full validation)
RUN_MODE = "demo"  # @param ["demo", "paper"]

In [ ]:
import sys
from pathlib import Path

# Ensure notebook utilities are importable
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

# Notebook utilities
from _utils import (
    NotebookConfig,
    print_config,
    print_repro_stamp,
    set_all_seeds,
    RunManager,
    setup_plotting,
    add_caption,
    POLICY_COLORS,
)

# Reckon core imports
from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy
from reckon.memory.record import MemoryRecord, RecordCost, CausalScore, UtilityEstimate

# Plotting
import matplotlib.pyplot as plt
setup_plotting()

In [ ]:
# Configuration
config = NotebookConfig(
    notebook_name="01_system_overview",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=1,
    budget_slots=50,
    task_name="educational",
    num_episodes=1,
    policy_name="evidence_per_byte",
)

print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)

## 2. Problem + Hypotheses

Understanding the relationship between mathematical theory and implementation is essential for:

1. **Verifying correctness:** Ensuring the code faithfully implements the intended algorithm
2. **Debugging failures:** Tracing unexpected behavior back to specific equations
3. **Extending the system:** Knowing where to modify code when changing the theory

**H1 (Symbol Correspondence):** Every symbol in the Reckon mathematical framework ($r$, $\mathcal{M}$, $B$, $\pi$, $u(r)$, $\Delta_r$, $c(r)$) has a direct, verifiable code counterpart.

**H2 (Operational Fidelity):** The data flow through the implementation matches the theoretical process: observations enter through adapters, get stored as records with utility scores, undergo policy-driven pruning, and emit structured event logs.

**H3 (Canonical Ordering):** The implementation uses $(\text{episode\_id}, \text{step}, \text{sequence})$ as the primary ordering key, not wall-clock timestamps, ensuring deterministic replay.

## 3. Minimal Theory

### 3.1 Record-removal attribution and causal interpretation

A removal comparison estimates a difference for its stated outcome, intervention and workload:

$$\Delta_r=\mathbb{E}[\mathrm{metric}\mid r\in\mathcal M]-\mathbb{E}[\mathrm{metric}\mid r\notin\mathcal M]$$

That comparison does not by itself identify a causal effect. The intervention, assumptions, controls and uncertainty matter. Native `Prober.probe()` in `src/reckon/metrics/probe.py` uses a caller-supplied paired runner; notebook 03 uses `CausalRecordEvaluator`, which removes the record at its first eligible read.

### 3.2 Evidence and decisions

The [evaluation manual](../docs/11-Evaluation-Metrics.md) distinguishes the evaluator’s beneficial `is_useful` result, Prober’s two-sided `Evidence.is_significant` and `Evidence.is_significant_adjusted` flags, and pair-bound debugger edge evidence. Each answers a different question. This notebook does not run those probes or test their statistical validity.

### 3.3 Canonical ordering fields

The intended ordering key is `(episode_id, step, sequence)`. The Rust event rows used here name their logical step `timestamp`; the native TypeScript comparator owns normalization and sorting in `viz-ui/packages/components/src/utils/canonicalOrder.ts`. The checks below inspect field presence, not full replay or comparator conformance.

See [Math Core](../docs/03-Math-Core.md) for the broader framework and the [architecture owner](../ARCHITECTURE.md) for current component responsibilities.


## 4. System Mapping

The following diagram illustrates the data flow from environment observations through the Causal Memory Manager to the debugger UI.

```mermaid
flowchart TB
    subgraph ENV["Environment / Conversation"]
        E1["MiniGrid | TextWorld | Dialogue | Custom"]
    end

    subgraph ADAPT["Adapters"]
        A1["notebooks/_utils/adapters/"]
        A2["Convert env obs → (payload, payload_type, tags)"]
    end

    subgraph CMM["Causal Memory Manager<br/>src/reckon/memory/manager.py"]
        direction TB

        subgraph OPS["Operations"]
            W["write()"]
            R["retrieve()"]
            P["prune()"]
            U["update_score()"]
        end

        POLICY["RetentionPolicy<br/>FIFO | LRU | Utility | Evidence per byte"]

        subgraph RUST["Rust Core"]
            STORE["RecordStore<br/>kernel/src/store.rs<br/>storage and priority index"]
            LOGGER["EventLogger<br/>kernel/src/logging.rs<br/>sequence emission"]
        end

        OPS --> POLICY
        POLICY --> STORE
        OPS --> LOGGER
    end

    subgraph LOGS["JSONL Logs"]
        L1["runs/date/notebook/hash/logs/events.jsonl"]
    end

    subgraph UI["Debugger UI<br/>viz-ui/packages/debugger/"]
        U1["Timeline View"]
        U2["Causal Graph"]
        U3["Pareto Frontier"]
    end

    ENV --> ADAPT
    ADAPT --> CMM
    CMM -->|"explicit export"| LOGS
    LOGS --> UI
```


## 5. Experiment Design

This notebook examines selected theory-to-code correspondences through instantiated objects and a synthetic trace.

**Unit of analysis:** Individual code artifacts corresponding to mathematical symbols.

**Validation approach:**

1. Enumerate the mathematical symbols selected for this walkthrough
2. For each symbol, identify the corresponding code location
3. Instantiate actual objects and verify field correspondence
4. Trace a complete data flow through the system

**Checks to inspect:**

- The selected symbols have identifiable code counterparts
- The synthetic trace follows writes, storage and pruning
- Retained slots fit this trace’s configured allowance
- The retained event rows contain the ordering fields

These checks do not establish full theoretical fidelity, every budget case or deterministic replay; those need their own tests.


## 6. Demo Run: Symbol-to-Code Mapping

The following table establishes the correspondence between mathematical notation and implementation. Each row can be verified by inspecting the referenced source file.

In [ ]:
# Display the symbol-to-code mapping as a formatted table
symbol_mapping = [
    ("r", "Memory record", "MemoryRecord", "dataclass", "memory/record.py"),
    ("M", "Memory store", "CausalMemoryManager._records", "dict", "memory/manager.py"),
    ("B", "Budget constraint", "MemoryBudget", "dataclass", "core/resources.py"),
    ("π", "Retention policy", "RetentionPolicy", "ABC", "memory/policies.py"),
    ("u(r)", "Utility estimate", "MemoryRecord.utility.value", "float", "memory/record.py"),
    ("Δ_r", "Ablation delta", "CausalScore.ablation_delta", "Optional[float]", "memory/record.py"),
    ("c(r)", "Record cost", "RecordCost.bytes", "int", "memory/record.py"),
    ("t", "Timestep", "step (canonical)", "int", "events"),
    ("ε", "Episode", "episode_id", "str", "records/events"),
]

print("Symbol-to-Code Mapping")
print("=" * 90)
print(f"{'Symbol':<8} {'Meaning':<20} {'Code Field':<30} {'Type':<15} {'Location'}")
print("-" * 90)
for symbol, meaning, code_field, dtype, location in symbol_mapping:
    print(f"{symbol:<8} {meaning:<20} {code_field:<30} {dtype:<15} {location}")
print("=" * 90)

In [ ]:
# Demonstrate the mapping with actual instantiated objects
print("\nVerification: Instantiating actual objects\n")

# Create a CMM instance and write a record
cmm = CausalMemoryManager(
    budget=MemoryBudget(max_slots=10),
    episode_id="verification-demo"
)
cmm.set_step(5)

record_id = cmm.write(
    payload={"color": "red", "type": "key"},
    payload_type=PayloadType.CUE,
    tags=["cue", "key", "red"],
    initial_utility=0.8,
)

# Retrieve the record and display field correspondence
r = cmm.get_record(record_id)
verification_record = r  # Keep this record distinct from later trace-loop variables.

print("MemoryRecord instance fields:")
print(f"  r.id              = {r.id}")
print(f"  r.episode_id      = {r.episode_id}       # ε in theory")
print(f"  r.timestamp       = {r.timestamp}                  # t (step) in theory")
print(f"  r.payload_type    = {r.payload_type}")
print(f"  r.cost.bytes      = {r.cost.bytes}                # c(r) in theory")
print(f"  r.utility.value   = {r.utility.value}              # u(r) in theory")
print(f"  r.causal_score.ablation_delta = {r.causal_score.ablation_delta}  # Δ_r in theory")
print(f"  r.tags            = {r.tags}")

### 6.1 Policy Comparison

The retention policy $\pi$ determines which records to prune when the budget is exceeded. Each policy implements the abstract `select_for_pruning()` method.

In [ ]:
# Display available policies and their characteristics
policies = [
    ("FIFO", FIFOPolicy(), "Oldest records first", "Simple baseline, ignores utility"),
    ("LRU", LRUPolicy(), "Least recently accessed", "Access-pattern workloads"),
    ("Utility", UtilityPolicy(), "Lowest utility/cost ratio", "Value-aware retention"),
    ("Evidence per byte", EvidencePerBytePolicy(evidence_weight=0.5), "Utility + supplied evidence per byte", "When Δ_r scores available"),
]

print("Available Retention Policies")
print("=" * 80)
print(f"{'Policy':<10} {'Prune Criterion':<25} {'Best For':<30} {'Reason'}")
print("-" * 80)
for name, policy, criterion, use_case in policies:
    print(f"{name:<10} {criterion:<25} {use_case:<30} {policy.reason()}")
print("=" * 80)

## 7. Full Run: Data Flow Trace

This section traces the complete data flow through the system, demonstrating how records enter, get stored, trigger pruning when budget is exceeded, and produce event logs.

In [ ]:
# Trace the complete data flow through the system
print("Data Flow Trace")
print("=" * 70)

# Step 1: Initialize CMM with constrained budget
budget = MemoryBudget(max_slots=5, max_bytes=500)
cmm_trace = CausalMemoryManager(
    budget=budget,
    retention_policy=EvidencePerBytePolicy(evidence_weight=0.5),
    episode_id="trace-demo"
)
print(f"\n1. CMM initialized")
print(f"   Budget: {budget.max_slots} slots, {budget.max_bytes} bytes")
print(f"   Policy: EvidencePerBytePolicy(w=0.5)")

# Step 2: Write records that will exceed budget
print(f"\n2. Writing 7 records (exceeds budget of 5 slots)")
print("-" * 70)

for i in range(7):
    cmm_trace.set_step(i)
    rid = cmm_trace.write(
        payload={"index": i, "data": "x" * 50},
        payload_type=PayloadType.OBSERVATION,
        tags=[f"record_{i}"],
        initial_utility=0.1 * i,  # Increasing utility values
    )
    slots_used = cmm_trace.current_usage_slots
    status = "PRUNED" if slots_used < i + 1 else "STORED"
    print(f"   Step {i}: {rid[:8]}... utility={0.1*i:.1f}  slots={slots_used}/5  [{status}]")

# Step 3: Examine final state
print(f"\n3. Final state after pruning")
print("-" * 70)
records = cmm_trace.get_all_records()
print(f"   Records retained: {len(records)}")
for r in sorted(records, key=lambda x: x.timestamp):
    print(f"   - {r.tags[0]}: step={r.timestamp}, utility={r.utility.value:.1f}")

# Step 4: Examine event log
print(f"\n4. Event log summary")
print("-" * 70)
events = cmm_trace.get_event_log()
event_counts = {}
for e in events:
    etype = e.get('op_type', 'unknown')
    event_counts[etype] = event_counts.get(etype, 0) + 1

print(f"   Total events: {len(events)}")
for etype, count in sorted(event_counts.items()):
    print(f"   - {etype}: {count}")

## 8. Results

These checks cover the instantiated record, local slot usage and native event fields. They do not prove every theoretical symbol, general hard-budget admission or canonical replay.

In [ ]:
# Summarize bounded source-example checks, not full theoretical conformance.
print("Field and trace checks")
print("=" * 70)

v = verification_record
h1_pass = v is not None and all([
    v.id is not None,
    v.episode_id == "verification-demo",
    v.timestamp >= 0,
    v.utility.value >= 0,
    v.cost.bytes > 0,
])
print(f"Selected record fields: {'PRESENT' if h1_pass else 'MISSING'}")

h2_pass = all([
    cmm_trace.current_usage_slots <= budget.max_slots,
    len(events) > 0,
    'write' in event_counts or 'prune' in event_counts,
])
print(f"Local trace/slot check: {'PASS' if h2_pass else 'FAIL'}")
print(f"Retained slots: {cmm_trace.current_usage_slots}/{budget.max_slots}; events={len(events)}")

h3_pass = bool(events) and all(
    all(e.get(key) is not None for key in ("episode_id", "timestamp", "sequence"))
    for e in events
)
print(f"Canonical event fields: {'PRESENT' if h3_pass else 'MISSING'}")
print("Field presence does not test comparator ordering or deterministic replay.")
all_pass = h1_pass and h2_pass and h3_pass
print(f"Overall bounded checks: {'PASS' if all_pass else 'FAIL'}")

In [ ]:
# Visualize the data flow as a simple diagram
fig, ax = plt.subplots(figsize=(10, 4))

# Plot event counts as horizontal bars
event_types = list(event_counts.keys())
counts = list(event_counts.values())
colors = [POLICY_COLORS.get('evidence_per_byte', '#8B5CF6')] * len(event_types)

y_pos = range(len(event_types))
ax.barh(y_pos, counts, color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(event_types)
ax.set_xlabel('Count')
ax.set_title('Event Log Composition')

# Add count labels
for i, count in enumerate(counts):
    ax.text(count + 0.1, i, str(count), va='center')

add_caption(
    ax,
    "Distribution of event types captured during data flow trace",
    n=len(events),
    mode=RUN_MODE,
    budget=budget.max_slots,
)

plt.tight_layout()
plt.show()

## 9. Debugger Walkthrough

The export cell saves this trace's collected events to `logs/events.jsonl`. Follow the [native debugger quick start](../src/reckon/viz/README.md#quick-start) for actual import and run selection.

Compare write/prune record IDs and logical steps with the records printed in this notebook. Read events carry hit IDs; they do not preserve the original query. A write event alone does not include its initial utility or payload. This trace does not emit pair-bound MEASURED edges or a complete policy-comparison dataset; use the [debugger design](../docs/10-Debugger-Visualization.md) to distinguish the inputs those views need.

## 10. Limitations + Next Steps

### Limitations

1. **No statistical testing:** This notebook demonstrates correspondence, not performance. For statistical evaluation of causal claims, see notebook 03.

2. **Simplified data flow:** The trace uses synthetic data rather than actual environment observations. Real workloads involve adapters that convert environment-specific formats.

3. **Single policy:** The trace demonstrates EvidencePerBytePolicy only. Comparative analysis across policies is covered in notebook 02.

### Next Steps

- [ ] **Notebook 02:** Compare FIFO, LRU, Utility, and Evidence per byte policies on KeyDoor task
- [ ] **Notebook 03:** Verify the causal record contract with ablation experiments
- [ ] **Notebook 05:** Integrate with MiniGrid environments using adapters
- [ ] **Notebook 06:** Integrate with TextWorld environments for text-based tasks

## Export Artifacts

In [ ]:
# Save all artifacts
run = RunManager(config)
run.save_config()

# Save event logs from the trace
run.save_logs(events)

# Save summary metrics
summary_metrics = {
    "symbols_mapped": len(symbol_mapping),
    "records_final": len(records),
    "events_logged": len(events),
    "budget_slots": budget.max_slots,
    "h1_pass": h1_pass,
    "h2_pass": h2_pass,
    "canonical_fields_present": h3_pass,
}
run.save_metrics(summary_metrics)

# Write summary report
run.write_summary(
    key_metrics=summary_metrics,
    notes="Educational notebook demonstrating theory-to-code correspondence."
)

run.print_artifacts()

## Further Reading

- [docs/03-Math-Core.md](../docs/03-Math-Core.md) — Complete mathematical foundations
- [docs/11-Evaluation-Metrics.md](../docs/11-Evaluation-Metrics.md) — Causal Record Contract specification
- [docs/12-Reproducibility.md](../docs/12-Reproducibility.md) — Canonical ordering and deterministic replay
- [ARCHITECTURE.md](../ARCHITECTURE.md) — System design overview and component relationships